In [1]:
# ==============================================================================
# CONFIGURATION ET IMPORTS
# ==============================================================================

import json
import numpy as np
import pandas as pd
from pandas import json_normalize
import os
import re
import gc
import logging
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.feature_selection import VarianceThreshold
import joblib

import torch
import torch.nn.functional as F
from transformers import CamembertTokenizer, CamembertModel
from tqdm.auto import tqdm

# Logging configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA RTX 4000 Ada Generation


In [2]:
# ==============================================================================
# CONFIGURATION DATACLASS
# ==============================================================================

@dataclass
class PreprocessingConfig:
    """Configuration centralisée pour le preprocessing."""
    # Paths
    data_dir: str = "./data"
    embedding_dir: str = "./data/embeddings"
    feature_dir: str = "./data/features"
    
    # Columns
    text_col: str = "full_text"
    desc_col: str = "user.description"
    id_col: str = "challenge_id"
    
    # Model settings
    model_name: str = "camembert-base"
    max_length: int = 128
    batch_size: int = 32
    
    # Multi-layer embedding settings
    use_multi_layer: bool = True
    layers_to_use: Tuple[int, ...] = (-1, -2, -3, -4)  # Last 4 layers
    pooling_strategy: str = "attention"  # "cls", "mean", "attention"
    
    # Mixed precision
    use_fp16: bool = True
    
    def __post_init__(self):
        os.makedirs(self.data_dir, exist_ok=True)
        os.makedirs(self.embedding_dir, exist_ok=True)
        os.makedirs(self.feature_dir, exist_ok=True)

config = PreprocessingConfig()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

2025-12-09 12:11:27,446 - INFO - Using device: cuda


## 1. Chargement des données

In [3]:
# ==============================================================================
# DATA LOADING
# ==============================================================================

def load_jsonl_data(filepath: str) -> pd.DataFrame:
    """Charge et normalise un fichier JSONL."""
    logger.info(f"Loading {filepath}...")
    df = pd.read_json(filepath, lines=True)
    df = json_normalize(df.to_dict(orient="records"))
    logger.info(f"Loaded {len(df)} records with {len(df.columns)} columns")
    return df

def extract_full_text(row: pd.Series) -> str:
    """Extrait le texte complet (extended si disponible)."""
    txt = row.get("text", "")
    extended = row.get("extended_tweet.full_text", np.nan)
    if pd.notna(extended) and extended:
        return str(extended)
    return str(txt) if pd.notna(txt) else ""

# Load data
train_df = load_jsonl_data("data/train.jsonl")
kaggle_df = load_jsonl_data("data/kaggle_test.jsonl")

# Separate features and labels
y_train = train_df["label"].values
X_train = train_df.drop("label", axis=1).copy()
X_kaggle = kaggle_df.copy()

# Extract full text
X_train[config.text_col] = X_train.apply(extract_full_text, axis=1)
X_kaggle[config.text_col] = X_kaggle.apply(extract_full_text, axis=1)

# Save labels
np.save(os.path.join(config.data_dir, "y_train.npy"), y_train)
logger.info(f"Labels distribution: {np.bincount(y_train)}")

print(f"\n📊 Data summary:")
print(f"   Train: {len(X_train)} samples")
print(f"   Test:  {len(X_kaggle)} samples")
print(f"   Labels: {np.sum(y_train == 0)} observers, {np.sum(y_train == 1)} influencers")

2025-12-09 12:11:27,484 - INFO - Loading data/train.jsonl...
2025-12-09 12:11:48,221 - INFO - Loaded 154914 records with 193 columns
2025-12-09 12:11:48,226 - INFO - Loading data/kaggle_test.jsonl...
2025-12-09 12:11:48,221 - INFO - Loaded 154914 records with 193 columns
2025-12-09 12:11:48,226 - INFO - Loading data/kaggle_test.jsonl...
2025-12-09 12:12:02,221 - INFO - Loaded 103380 records with 191 columns
2025-12-09 12:12:02,221 - INFO - Loaded 103380 records with 191 columns
2025-12-09 12:12:06,487 - INFO - Labels distribution: [82674 72240]
2025-12-09 12:12:06,487 - INFO - Labels distribution: [82674 72240]



📊 Data summary:
   Train: 154914 samples
   Test:  103380 samples
   Labels: 82674 observers, 72240 influencers


## 2. Feature Engineering - Text Features

In [4]:
# ==============================================================================
# TEXTUAL FEATURE EXTRACTION
# ==============================================================================

try:
    import emoji
    HAS_EMOJI = True
except ImportError:
    HAS_EMOJI = False
    logger.warning("emoji package not installed, using fallback")

class TextFeatureExtractor:
    """Extracteur de features textuelles optimisé."""
    
    # Patterns pré-compilés pour performance
    HASHTAG_PATTERN = re.compile(r'#\w+', re.UNICODE)
    MENTION_PATTERN = re.compile(r'@\w+', re.UNICODE)
    URL_PATTERN = re.compile(r'https?://\S+', re.UNICODE)
    RT_PATTERN = re.compile(r'(^RT @|\bRT\b|\bQT[:]?\b|RT\s@)', re.IGNORECASE)
    MEDIA_PATTERN = re.compile(r'photo|image|vid[eé]o|video|pic|gif', re.IGNORECASE)
    EMOJI_PATTERN = re.compile(r'[\U0001F300-\U0001F6FF\U0001F900-\U0001F9FF\u2600-\u27BF]')
    
    # Mots clés pour détection
    CTA_KEYWORDS = ['suivez', 'abonnez', "s'abonner", 'follow', 'subscribe', 
                    'retweet to win', 'like & follow', 'rt pour', 'rt if']
    PROMO_KEYWORDS = ['nouveau', 'article', 'vidéo', 'video', 'lien', 'link', 
                      'disponible', 'promo', 'promotion', 'exclusif', 'découvrez']
    INFLUENCER_KEYWORDS = ['partenariat', 'sponsorisé', 'ad', 'pub', 'collab',
                           'concours', 'giveaway', 'code promo']
    
    @staticmethod
    def count_emojis(text: str) -> int:
        if not isinstance(text, str):
            return 0
        if HAS_EMOJI and hasattr(emoji, 'EMOJI_DATA'):
            return sum(1 for ch in text if ch in emoji.EMOJI_DATA)
        return len(TextFeatureExtractor.EMOJI_PATTERN.findall(text))
    
    @classmethod
    def extract_all(cls, texts: pd.Series) -> pd.DataFrame:
        """Extrait toutes les features textuelles."""
        texts = texts.astype(str)
        texts_lower = texts.str.lower()
        
        features = pd.DataFrame(index=texts.index)
        
        # Basic counts
        features['tweet_length'] = texts.str.len()
        features['word_count'] = texts.str.split().str.len().fillna(0)
        features['char_per_word'] = (features['tweet_length'] / features['word_count'].replace(0, 1)).fillna(0)
        
        # Hashtags
        features['hashtag_count'] = texts.str.count(cls.HASHTAG_PATTERN)
        features['is_hashtag_heavy'] = (features['hashtag_count'] > 3).astype(int)
        
        # Mentions
        features['mention_count'] = texts.str.count(cls.MENTION_PATTERN)
        features['is_mention_heavy'] = (features['mention_count'] > 2).astype(int)
        
        # URLs
        features['url_count'] = texts.str.count(cls.URL_PATTERN)
        features['has_url'] = (features['url_count'] > 0).astype(int)
        
        # Emojis
        features['emoji_count'] = texts.apply(cls.count_emojis)
        features['is_emoji_heavy'] = (features['emoji_count'] > 5).astype(int)
        
        # Punctuation
        features['exclamation_count'] = texts.str.count('!')
        features['question_count'] = texts.str.count('\\?')
        features['has_multiple_exclamations'] = (features['exclamation_count'] > 1).astype(int)
        
        # Uppercase ratio
        alpha_chars = texts.str.replace(r'[^a-zA-Z]', '', regex=True)
        upper_chars = texts.str.replace(r'[^A-Z]', '', regex=True)
        features['uppercase_ratio'] = (upper_chars.str.len() / alpha_chars.str.len().replace(0, 1)).fillna(0)
        
        # Content type detection
        features['has_rt_qt'] = texts.str.contains(cls.RT_PATTERN, na=False).astype(int)
        features['is_reply'] = texts.str.strip().str.startswith('@').astype(int)
        features['has_media_reference'] = texts.str.contains(cls.MEDIA_PATTERN, na=False).astype(int)
        
        # Keyword detection
        features['has_call_to_action'] = texts_lower.apply(
            lambda x: int(any(kw in x for kw in cls.CTA_KEYWORDS))
        )
        features['has_self_promotion'] = texts_lower.apply(
            lambda x: int(any(kw in x for kw in cls.PROMO_KEYWORDS))
        )
        features['has_influencer_keywords'] = texts_lower.apply(
            lambda x: int(any(kw in x for kw in cls.INFLUENCER_KEYWORDS))
        )
        
        # Engagement indicators
        features['engagement_score'] = (
            features['has_call_to_action'] * 2 +
            features['has_self_promotion'] +
            features['has_url'] +
            features['hashtag_count'].clip(upper=5) * 0.5
        )
        
        return features

# Extract text features
logger.info("Extracting text features...")
train_text_features = TextFeatureExtractor.extract_all(X_train[config.text_col])
kaggle_text_features = TextFeatureExtractor.extract_all(X_kaggle[config.text_col])

print(f"\n📝 Text features extracted: {train_text_features.shape[1]} features")
train_text_features.head()

2025-12-09 12:12:06,537 - INFO - Extracting text features...



📝 Text features extracted: 22 features


,tweet_length,word_count,char_per_word,hashtag_count,is_hashtag_heavy,mention_count,is_mention_heavy,url_count,has_url,emoji_count,...,question_count,has_multiple_exclamations,uppercase_ratio,has_rt_qt,is_reply,has_media_reference,has_call_to_action,has_self_promotion,has_influencer_keywords,engagement_score
0,23,4,5.750000,0,0,0,0,0,0,0,...,0,0,0.066667,0,0,0,0,0,0,0.0
1,114,16,7.125000,0,0,0,0,0,0,0,...,0,0,0.044944,0,0,0,0,0,0,0.0
2,296,40,7.400000,0,0,4,1,0,0,0,...,0,0,0.025532,0,1,1,0,0,1,0.0
3,205,30,6.833333,1,0,2,0,1,1,1,...,0,0,0.142857,0,0,0,0,0,0,1.5
4,263,45,5.844444,0,0,3,1,0,0,0,...,0,0,0.029851,0,1,0,0,0,1,0.0


## 3. Feature Engineering - Structured Features

In [5]:
# ==============================================================================
# CLEAN AND SYNC COLUMNS
# ==============================================================================

def drop_complex_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Supprime les colonnes complexes (nested objects)."""
    bad_cols = []
    for col in df.columns:
        if df[col].apply(lambda x: isinstance(x, (dict, list))).any():
            bad_cols.append(col)
    logger.info(f"Dropping {len(bad_cols)} complex columns")
    return df.drop(columns=bad_cols)

# Clean dataframes
X_train_clean = drop_complex_columns(X_train)
X_kaggle_clean = drop_complex_columns(X_kaggle)

# Synchronize columns
common_cols = list(set(X_train_clean.columns) & set(X_kaggle_clean.columns))
for col in [config.text_col, config.id_col]:
    if col in common_cols:
        common_cols.remove(col)

# Define column types
# ⚠️ NOTE: user.followers_count et user.friends_count N'EXISTENT PAS dans les données!
# On utilise uniquement les colonnes réellement disponibles
NUMERIC_COLS = [
    # quoted_status.user features (celles-ci EXISTENT dans quoted_status)
    "quoted_status.user.favourites_count", "quoted_status.favorite_count",
    "quoted_status.reply_count", "quoted_status.user.friends_count",
    "quoted_status.quote_count", "quoted_status.user.listed_count",
    "quoted_status.retweet_count", "quoted_status.user.followers_count",
    # user features (SANS followers_count et friends_count - N'EXISTENT PAS!)
    "user.listed_count", "user.favourites_count", "user.statuses_count",
    # Tweet engagement
    "retweet_count", "favorite_count", "reply_count", "quote_count"
]
NUMERIC_COLS = [c for c in NUMERIC_COLS if c in common_cols]

CATEGORICAL_COLS = [
    "is_quote_status", "truncated", "possibly_sensitive",
    "user.geo_enabled", "user.is_translator", "user.default_profile",
    "user.profile_use_background_image", "user.translator_type"
]
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in common_cols]

# ==============================================================================
# ⭐ FEATURE ENGINEERING AMÉLIORÉ (basé sur feature_engineering.ipynb)
# ==============================================================================

def extract_source_device(source_str):
    """Extrait le type d'appareil depuis la source Twitter.
    
    Feature très discriminante découverte dans feature_engineering.ipynb!
    """
    if pd.isna(source_str):
        return 'unknown'
    source = str(source_str).lower()
    if 'iphone' in source:
        return 'iphone'
    elif 'android' in source:
        return 'android'
    elif 'tweetdeck' in source:
        return 'tweetdeck'
    elif 'web' in source or 'browser' in source:
        return 'web'
    elif any(x in source for x in ['buffer', 'hootsuite', 'socialflow', 'sprout', 'dlvr.it']):
        return 'bot_scheduler'
    else:
        return 'other'

def add_user_features(df: pd.DataFrame) -> pd.DataFrame:
    """Ajoute des features utilisateur dérivées.
    
    ⭐ AMÉLIORÉ avec les TOP features de feature_engineering.ipynb:
    - user_description_length (importance=736)
    - tweets_per_favourites (importance=699)
    - user_statuses_count (importance=639)
    - user_listed_count (importance=607)
    - listed_per_status (importance=553)
    - log_user_listed (corr=0.606)
    - total_engagement (très discriminant)
    - source_device (iphone/android/tweetdeck/bot)
    """
    df = df.copy()
    
    # === Colonnes de base ===
    listed = df.get('user.listed_count', pd.Series([0]*len(df), index=df.index)).fillna(0)
    statuses = df.get('user.statuses_count', pd.Series([1]*len(df), index=df.index)).fillna(1).replace(0, 1)
    favourites = df.get('user.favourites_count', pd.Series([0]*len(df), index=df.index)).fillna(0)
    
    # === ⭐ TOP Features brutes (importance LightGBM élevée) ===
    df['user_statuses_count'] = statuses
    df['user_favourites_count'] = favourites
    df['user_listed_count'] = listed
    
    # === ⭐ Ratios très discriminants ===
    
    # Listed per status ratio (importance=553, Observer=0, Influencer=25 median)
    df['user_listed_per_status'] = listed / statuses
    df['user_listed_per_status'] = df['user_listed_per_status'].clip(upper=1)
    
    # Tweets per favourites ratio (importance=699!)
    df['user_tweets_per_favourites'] = statuses / (favourites + 1)
    df['user_tweets_per_favourites'] = df['user_tweets_per_favourites'].clip(upper=100)
    
    # === ⭐ Log transforms (corr très élevée) ===
    df['log_user_statuses'] = np.log1p(statuses)      # corr=0.439
    df['log_user_favourites'] = np.log1p(favourites)
    df['log_user_listed'] = np.log1p(listed)           # corr=0.606 ⭐
    
    # === ⭐ Total Engagement (feature composite très discriminante) ===
    retweet = df.get('retweet_count', pd.Series([0]*len(df), index=df.index)).fillna(0)
    favorite = df.get('favorite_count', pd.Series([0]*len(df), index=df.index)).fillna(0)
    reply = df.get('reply_count', pd.Series([0]*len(df), index=df.index)).fillna(0)
    quote = df.get('quote_count', pd.Series([0]*len(df), index=df.index)).fillna(0)
    
    df['total_engagement'] = retweet + favorite + reply + quote
    df['log_total_engagement'] = np.log1p(df['total_engagement'])
    df['log_retweet_count'] = np.log1p(retweet)
    df['log_favorite_count'] = np.log1p(favorite)
    
    # === ⭐ Source Device (très discriminant!) ===
    if 'source' in df.columns:
        df['source_device'] = df['source'].apply(extract_source_device)
    else:
        df['source_device'] = 'unknown'
    
    # === ⭐ Features binaires très discriminantes ===
    
    # Has banner (Observer=73%, Influencer=92%)
    if 'user.profile_banner_url' in df.columns:
        df['user_has_banner'] = df['user.profile_banner_url'].notna().astype(int)
    else:
        df['user_has_banner'] = 0
    
    # Has location (Observer=58%, Influencer=75%)
    if 'user.location' in df.columns:
        df['user_has_location'] = (df['user.location'].notna() & (df['user.location'] != '')).astype(int)
    else:
        df['user_has_location'] = 0
    
    # Has URL (Observer=16%, Influencer=56%)
    if 'user.url' in df.columns:
        df['user_has_url'] = df['user.url'].notna().astype(int)
    else:
        df['user_has_url'] = 0
    
    # Has description (importance=736!)
    if 'user.description' in df.columns:
        df['user_has_description'] = (df['user.description'].notna() & (df['user.description'] != '')).astype(int)
        df['user_desc_length'] = df['user.description'].fillna('').str.len()
        df['user_has_long_desc'] = (df['user_desc_length'] > 100).astype(int)
    else:
        df['user_has_description'] = 0
        df['user_desc_length'] = 0
        df['user_has_long_desc'] = 0
    
    # Default profile flags
    if 'user.default_profile' in df.columns:
        df['user_default_profile'] = df['user.default_profile'].fillna(False).astype(int)
    else:
        df['user_default_profile'] = 0
    
    if 'user.default_profile_image' in df.columns:
        df['user_default_profile_image'] = df['user.default_profile_image'].fillna(False).astype(int)
    else:
        df['user_default_profile_image'] = 0
    
    # === ⭐ Is reply (très discriminant: Observer=39%, Influencer=19%) ===
    if 'in_reply_to_status_id' in df.columns:
        df['is_reply_real'] = df['in_reply_to_status_id'].notna().astype(int)
    else:
        df['is_reply_real'] = 0
    
    # Is quote status
    if 'is_quote_status' in df.columns:
        df['is_quote_status_flag'] = df['is_quote_status'].fillna(False).astype(int)
    else:
        df['is_quote_status_flag'] = 0
    
    # Has quoted status (le tweet cite un autre)
    if 'quoted_status.id' in df.columns:
        df['has_quoted_status'] = df['quoted_status.id'].notna().astype(int)
    else:
        df['has_quoted_status'] = 0
    
    return df

# Apply user features
X_train_clean = add_user_features(X_train_clean)
X_kaggle_clean = add_user_features(X_kaggle_clean)

# ⭐ Update derived cols avec TOUTES les TOP features
DERIVED_COLS = [
    # TOP features brutes (importance LightGBM)
    'user_statuses_count', 'user_favourites_count', 'user_listed_count',
    # Ratios discriminants
    'user_listed_per_status', 'user_tweets_per_favourites',
    # Log transforms (corr élevée)
    'log_user_statuses', 'log_user_favourites', 'log_user_listed',
    # Engagement
    'total_engagement', 'log_total_engagement', 'log_retweet_count', 'log_favorite_count',
    # Binary features
    'user_has_banner', 'user_has_location', 'user_has_url',
    'user_has_description', 'user_desc_length', 'user_has_long_desc',
    'user_default_profile', 'user_default_profile_image',
    'is_reply_real', 'is_quote_status_flag', 'has_quoted_status'
]

# Source device sera encodé en one-hot séparément
SOURCE_DEVICE_COL = ['source_device']

print(f"\n📊 Structured features (AMÉLIORÉES avec feature_engineering.ipynb):")
print(f"   Numeric: {len(NUMERIC_COLS)} columns")
print(f"   Categorical: {len(CATEGORICAL_COLS)} columns")
print(f"   Derived: {len(DERIVED_COLS)} columns ⭐")
print(f"   Source device: encodage one-hot")

2025-12-09 12:12:17,835 - INFO - Dropping 36 complex columns
2025-12-09 12:12:19,700 - INFO - Dropping 35 complex columns
2025-12-09 12:12:19,700 - INFO - Dropping 35 complex columns



📊 Structured features (AMÉLIORÉES avec feature_engineering.ipynb):
   Numeric: 15 columns
   Categorical: 8 columns
   Derived: 23 columns ⭐
   Source device: encodage one-hot


In [6]:
# ==============================================================================
# SKLEARN PREPROCESSING PIPELINE (AMÉLIORÉ)
# ==============================================================================

# Combine all structured features
all_numeric_cols = NUMERIC_COLS + DERIVED_COLS

# ⭐ Add source_device to categorical columns
all_categorical_cols = CATEGORICAL_COLS + SOURCE_DEVICE_COL

# Fill NaN values for numeric columns
for col in all_numeric_cols:
    if col in X_train_clean.columns:
        X_train_clean[col] = pd.to_numeric(X_train_clean[col], errors='coerce').fillna(0)
    if col in X_kaggle_clean.columns:
        X_kaggle_clean[col] = pd.to_numeric(X_kaggle_clean[col], errors='coerce').fillna(0)

# ⚠️ FIX: Convert categorical columns to strings (avoid mixed str/float from NaN)
for col in all_categorical_cols:
    if col in X_train_clean.columns:
        X_train_clean[col] = X_train_clean[col].fillna("missing").astype(str)
    if col in X_kaggle_clean.columns:
        X_kaggle_clean[col] = X_kaggle_clean[col].fillna("missing").astype(str)

print(f"📊 Columns prepared (AMÉLIORÉ):")
print(f"   Numeric: {len(all_numeric_cols)} (includes TOP features from feature_engineering)")
print(f"   Categorical: {len(all_categorical_cols)} (includes source_device)")

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler())  # More robust to outliers
        ]), all_numeric_cols),
        ("cat", Pipeline([
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), all_categorical_cols),
    ],
    remainder="drop",
    n_jobs=-1
)

# Fit and transform
logger.info("Fitting preprocessing pipeline...")
X_train_struct = preprocessor.fit_transform(X_train_clean)
X_kaggle_struct = preprocessor.transform(X_kaggle_clean)

# Add text features
X_train_struct = np.hstack([X_train_struct, train_text_features.values])
X_kaggle_struct = np.hstack([X_kaggle_struct, kaggle_text_features.values])

# Save preprocessor
joblib.dump(preprocessor, os.path.join(config.feature_dir, "preprocessor.joblib"))

print(f"\n✅ Structured features processed:")
print(f"   Train shape: {X_train_struct.shape}")
print(f"   Test shape: {X_kaggle_struct.shape}")
print(f"\n⭐ Features ajoutées depuis feature_engineering.ipynb:")
print(f"   - user_statuses_count, user_favourites_count, user_listed_count")
print(f"   - total_engagement, log_total_engagement")
print(f"   - source_device (one-hot: iphone/android/tweetdeck/bot/web/other)")
print(f"   - user_default_profile_image, has_quoted_status")

2025-12-09 12:12:21,010 - INFO - Fitting preprocessing pipeline...


📊 Columns prepared (AMÉLIORÉ):
   Numeric: 38 (includes TOP features from feature_engineering)
   Categorical: 9 (includes source_device)

✅ Structured features processed:
   Train shape: (154914, 84)
   Test shape: (103380, 84)

⭐ Features ajoutées depuis feature_engineering.ipynb:
   - user_statuses_count, user_favourites_count, user_listed_count
   - total_engagement, log_total_engagement
   - source_device (one-hot: iphone/android/tweetdeck/bot/web/other)
   - user_default_profile_image, has_quoted_status

✅ Structured features processed:
   Train shape: (154914, 84)
   Test shape: (103380, 84)

⭐ Features ajoutées depuis feature_engineering.ipynb:
   - user_statuses_count, user_favourites_count, user_listed_count
   - total_engagement, log_total_engagement
   - source_device (one-hot: iphone/android/tweetdeck/bot/web/other)
   - user_default_profile_image, has_quoted_status


## 4. CamemBERT Embeddings (Multi-Layer)

In [7]:
# ==============================================================================
# MULTI-LAYER CAMEMBERT EMBEDDINGS
# ==============================================================================

class CamemBERTEmbedder:
    """Extracteur d'embeddings CamemBERT multi-couches."""
    
    def __init__(self, config: PreprocessingConfig, device: torch.device):
        self.config = config
        self.device = device
        
        logger.info(f"Loading {config.model_name}...")
        self.tokenizer = CamembertTokenizer.from_pretrained(config.model_name)
        self.model = CamembertModel.from_pretrained(
            config.model_name,
            output_hidden_states=config.use_multi_layer
        )
        self.model.to(device)
        self.model.eval()
        
        # Attention weights for pooling
        if config.pooling_strategy == "attention":
            hidden_size = self.model.config.hidden_size
            self.attention = torch.nn.Linear(hidden_size, 1).to(device)
            torch.nn.init.xavier_uniform_(self.attention.weight)
        
        logger.info(f"Model loaded on {device}")
    
    def _pool_sequence(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """Pool sequence selon la stratégie choisie."""
        if self.config.pooling_strategy == "cls":
            return hidden_states[:, 0, :]
        
        elif self.config.pooling_strategy == "mean":
            # Mean pooling with attention mask
            mask = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
            sum_hidden = torch.sum(hidden_states * mask, dim=1)
            sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
            return sum_hidden / sum_mask
        
        elif self.config.pooling_strategy == "attention":
            # Attention-weighted pooling
            mask = attention_mask.unsqueeze(-1).float()
            attn_scores = self.attention(hidden_states)  # (B, L, 1)
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
            attn_weights = F.softmax(attn_scores, dim=1)
            return torch.sum(hidden_states * attn_weights, dim=1)
        
        else:
            raise ValueError(f"Unknown pooling strategy: {self.config.pooling_strategy}")
    
    @torch.no_grad()
    def embed_texts(
        self, 
        texts: List[str], 
        show_progress: bool = True
    ) -> np.ndarray:
        """Génère les embeddings pour une liste de textes."""
        embeddings = []
        batch_size = self.config.batch_size
        
        iterator = range(0, len(texts), batch_size)
        if show_progress:
            iterator = tqdm(iterator, desc="Embedding")
        
        for i in iterator:
            batch_texts = texts[i:i+batch_size]
            
            # Tokenize
            encoded = self.tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=self.config.max_length,
                return_tensors="pt"
            )
            input_ids = encoded['input_ids'].to(self.device)
            attention_mask = encoded['attention_mask'].to(self.device)
            
            # Forward pass avec mixed precision
            if self.config.use_fp16 and self.device.type == "cuda":
                with torch.cuda.amp.autocast():
                    outputs = self.model(input_ids, attention_mask=attention_mask)
            else:
                outputs = self.model(input_ids, attention_mask=attention_mask)
            
            # Extract embeddings
            if self.config.use_multi_layer:
                # Concatenate multiple layers
                layer_embeddings = []
                for layer_idx in self.config.layers_to_use:
                    hidden = outputs.hidden_states[layer_idx]
                    pooled = self._pool_sequence(hidden, attention_mask)
                    layer_embeddings.append(pooled)
                batch_emb = torch.cat(layer_embeddings, dim=-1)
            else:
                batch_emb = self._pool_sequence(outputs.last_hidden_state, attention_mask)
            
            embeddings.append(batch_emb.cpu().numpy().astype(np.float32))
            
            # Memory cleanup
            if i % (batch_size * 50) == 0:
                gc.collect()
                if self.device.type == "cuda":
                    torch.cuda.empty_cache()
        
        return np.vstack(embeddings)
    
    def get_embedding_dim(self) -> int:
        """Retourne la dimension des embeddings."""
        base_dim = self.model.config.hidden_size
        if self.config.use_multi_layer:
            return base_dim * len(self.config.layers_to_use)
        return base_dim

# Initialize embedder
embedder = CamemBERTEmbedder(config, device)
print(f"\n🤖 CamemBERT Embedder initialized")
print(f"   Embedding dimension: {embedder.get_embedding_dim()}")
print(f"   Pooling strategy: {config.pooling_strategy}")
print(f"   Multi-layer: {config.use_multi_layer} (layers: {config.layers_to_use})")

2025-12-09 12:12:24,769 - INFO - Loading camembert-base...
2025-12-09 12:12:26,315 - INFO - Model loaded on cuda
2025-12-09 12:12:26,315 - INFO - Model loaded on cuda



🤖 CamemBERT Embedder initialized
   Embedding dimension: 3072
   Pooling strategy: attention
   Multi-layer: True (layers: (-1, -2, -3, -4))


In [ ]:
# ==============================================================================
# GENERATE EMBEDDINGS
# ⚠️ Cette cellule peut prendre du temps - N'exécuter que si nécessaire
# ==============================================================================

REGENERATE_EMBEDDINGS = False  # Mettre à True pour régénérer

if REGENERATE_EMBEDDINGS:
    logger.info("Starting embedding generation...")
    
    # Prepare texts
    train_texts = X_train_clean[config.text_col].astype(str).tolist()
    train_desc = X_train_clean[config.desc_col].fillna("").astype(str).tolist()
    kaggle_texts = X_kaggle_clean[config.text_col].astype(str).tolist()
    kaggle_desc = X_kaggle_clean[config.desc_col].fillna("").astype(str).tolist()
    
    # Generate train text embeddings
    print("\n📝 Embedding TRAIN texts...")
    train_text_emb = embedder.embed_texts(train_texts)
    print(f"   Shape: {train_text_emb.shape}")
    
    # Generate train description embeddings
    print("\n👤 Embedding TRAIN descriptions...")
    train_desc_emb = embedder.embed_texts(train_desc)
    print(f"   Shape: {train_desc_emb.shape}")
    
    # Concatenate and save train embeddings immediately, then free memory
    train_full_emb = np.hstack([train_text_emb, train_desc_emb])
    np.save(os.path.join(config.embedding_dir, "X_train_multilayer_embeddings.npy"), train_full_emb)
    print(f"   ✅ Train embeddings saved: {train_full_emb.shape}")
    del train_text_emb, train_desc_emb, train_full_emb
    gc.collect()
    
    # Generate kaggle text embeddings
    print("\n📝 Embedding KAGGLE texts...")
    kaggle_text_emb = embedder.embed_texts(kaggle_texts)
    print(f"   Shape: {kaggle_text_emb.shape}")
    
    # Generate kaggle description embeddings
    print("\n👤 Embedding KAGGLE descriptions...")
    kaggle_desc_emb = embedder.embed_texts(kaggle_desc)
    print(f"   Shape: {kaggle_desc_emb.shape}")
    
    # Concatenate and save kaggle embeddings
    kaggle_full_emb = np.hstack([kaggle_text_emb, kaggle_desc_emb])
    np.save(os.path.join(config.embedding_dir, "X_kaggle_multilayer_embeddings.npy"), kaggle_full_emb)
    print(f"   ✅ Kaggle embeddings saved: {kaggle_full_emb.shape}")
    del kaggle_text_emb, kaggle_desc_emb, kaggle_full_emb
    
    print(f"\n✅ All embeddings saved!")
    
    # Cleanup model and GPU memory
    del embedder
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    
    print("🧹 Memory cleaned up")
else:
    print("⏭️ Skipping embedding generation (REGENERATE_EMBEDDINGS=False)")
    print("   Set REGENERATE_EMBEDDINGS=True to regenerate embeddings")

2025-12-09 12:12:26,346 - INFO - Starting embedding generation...



📝 Embedding TRAIN texts...


Embedding:   0%|          | 0/4842 [00:00<?, ?it/s]

   Shape: (154914, 3072)

👤 Embedding TRAIN descriptions...


Embedding:   0%|          | 0/4842 [00:00<?, ?it/s]

   Shape: (154914, 3072)

📝 Embedding KAGGLE texts...


Embedding:   0%|          | 0/3231 [00:00<?, ?it/s]

   Shape: (103380, 3072)

👤 Embedding KAGGLE descriptions...


Embedding:   0%|          | 0/3231 [00:00<?, ?it/s]

   Shape: (103380, 3072)

✅ All embeddings saved!
   Train: (154914, 6144)
   Kaggle: (103380, 6144)


## 5. Assemblage Final

In [ ]:
# ==============================================================================
# FINAL ASSEMBLY
# ==============================================================================

# Save structured features (only if they don't exist)
train_feat_path = os.path.join(config.feature_dir, "X_train_features.npy")
kaggle_feat_path = os.path.join(config.feature_dir, "X_kaggle_features.npy")

if not os.path.exists(train_feat_path):
    np.save(train_feat_path, X_train_struct)
    print(f"✅ Saved: {train_feat_path}")
else:
    print(f"⏭️ Skipped (already exists): {train_feat_path}")

if not os.path.exists(kaggle_feat_path):
    np.save(kaggle_feat_path, X_kaggle_struct)
    print(f"✅ Saved: {kaggle_feat_path}")
else:
    print(f"⏭️ Skipped (already exists): {kaggle_feat_path}")

logger.info(f"Structured features: {X_train_struct.shape}")

# 🧹 Clean up DataFrames - no longer needed
if 'X_train_clean' in dir():
    del X_train_clean
if 'X_kaggle_clean' in dir():
    del X_kaggle_clean
if 'train_text_features' in dir():
    del train_text_features
if 'kaggle_text_features' in dir():
    del kaggle_text_features
gc.collect()

# Load embeddings (use existing if not regenerated)
try:
    train_emb_path = os.path.join(config.embedding_dir, "X_train_multilayer_embeddings.npy")
    kaggle_emb_path = os.path.join(config.embedding_dir, "X_kaggle_multilayer_embeddings.npy")
    
    if os.path.exists(train_emb_path):
        X_train_emb = np.load(train_emb_path)
        X_kaggle_emb = np.load(kaggle_emb_path)
    else:
        # Fallback to single-layer embeddings
        X_train_emb = np.load(os.path.join(config.embedding_dir, "X_train_embeddings.npy"))
        X_kaggle_emb = np.load(os.path.join(config.embedding_dir, "X_kaggle_embeddings.npy"))
    
    logger.info(f"Embeddings loaded: {X_train_emb.shape}")
except FileNotFoundError as e:
    logger.error(f"Embedding files not found: {e}")
    logger.error("Please run the embedding generation cell first!")
    raise

2025-12-09 12:29:19,998 - INFO - Structured features saved: (154914, 84)
2025-12-09 12:29:20,898 - INFO - Embeddings loaded: (154914, 6144)


In [ ]:
# ==============================================================================
# CONCATENATE ALL FEATURES
# ==============================================================================

# Final concatenation: structured features + embeddings
X_train_full = np.hstack([X_train_struct, X_train_emb]).astype(np.float32)
X_kaggle_full = np.hstack([X_kaggle_struct, X_kaggle_emb]).astype(np.float32)

# 🧹 Free intermediate arrays
del X_train_struct, X_kaggle_struct, X_train_emb, X_kaggle_emb
gc.collect()

# Save final processed arrays (only if they don't exist to save disk space)
train_path = os.path.join(config.data_dir, "X_train_processed_multilayer.npy")
kaggle_path = os.path.join(config.data_dir, "X_kaggle_processed_multilayer.npy")

if not os.path.exists(train_path):
    np.save(train_path, X_train_full)
    print(f"✅ Saved: {train_path}")
else:
    print(f"⏭️ Skipped (already exists): {train_path}")

if not os.path.exists(kaggle_path):
    np.save(kaggle_path, X_kaggle_full)
    print(f"✅ Saved: {kaggle_path}")
else:
    print(f"⏭️ Skipped (already exists): {kaggle_path}")

# Calculate memory usage
train_size_gb = X_train_full.nbytes / (1024**3)
kaggle_size_gb = X_kaggle_full.nbytes / (1024**3)

print("\n" + "="*60)
print("✅ PREPROCESSING COMPLETE")
print("="*60)
print(f"\n📊 Final shapes:")
print(f"   X_train: {X_train_full.shape} ({train_size_gb:.2f} GB)")
print(f"   X_kaggle: {X_kaggle_full.shape} ({kaggle_size_gb:.2f} GB)")
print(f"\n💾 Files saved to:")
print(f"   {config.data_dir}/X_train_processed_multilayer.npy")
print(f"   {config.data_dir}/X_kaggle_processed_multilayer.npy")
print(f"   {config.feature_dir}/preprocessor.joblib")

NameError: name 'np' is not defined

In [ ]:
# ==============================================================================
# VALIDATION & MEMORY CLEANUP
# ==============================================================================

def validate_data(X: np.ndarray, name: str) -> bool:
    """Valide les données processées."""
    issues = []
    
    # Check for NaN
    nan_count = np.isnan(X).sum()
    if nan_count > 0:
        issues.append(f"Found {nan_count} NaN values")
    
    # Check for Inf
    inf_count = np.isinf(X).sum()
    if inf_count > 0:
        issues.append(f"Found {inf_count} Inf values")
    
    # Check for very large values
    large_count = (np.abs(X) > 1e6).sum()
    if large_count > 0:
        issues.append(f"Found {large_count} values > 1e6")
    
    if issues:
        print(f"⚠️ {name}: {', '.join(issues)}")
        return False
    else:
        print(f"✅ {name}: All checks passed")
        return True

print("\n🔍 Validating processed data...")
validate_data(X_train_full, "X_train")
validate_data(X_kaggle_full, "X_kaggle")

# Quick stats
print(f"\n📈 Data statistics:")
print(f"   Mean: {X_train_full.mean():.4f}")
print(f"   Std:  {X_train_full.std():.4f}")
print(f"   Min:  {X_train_full.min():.4f}")
print(f"   Max:  {X_train_full.max():.4f}")

# 🧹 Final memory cleanup - free everything except final arrays
print("\n🧹 Final memory cleanup...")
del X_train_full, X_kaggle_full
gc.collect()

# Show disk usage
import subprocess
result = subprocess.run(['du', '-sh', config.data_dir], capture_output=True, text=True)
print(f"\n💾 Disk usage for {config.data_dir}: {result.stdout.strip()}")


🔍 Validating processed data...


NameError: name 'X_train_full' is not defined